# Last Modified: 2026-04-14 16:15:00

---

# LFP Battery SOH Preprocessing Pipeline (Optimized Batch Version)

This notebook implements a physically-consistent preprocessing framework for the entire dataset:
1. **Phase 1**: Global Physical Cleaning (Unit conversion and noise smoothing for ALL cells)
2. **Phase 2**: Scenario-based Slicing Definition
3. **Phase 2.1**: **Global Slicing Loop (Generate & Cache Raw Segment Pool)**
4. **Phase 3**: 40D HI Extraction Logic Definition
5. **Phase 4**: **Feature Extraction & Scaling (Reuse Sliced Data)**
6. **Phase 5**: Global Tensor Saving

In [ ]:
import numpy as np
import pandas as pd
import gc
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
from pathlib import Path
from tqdm.notebook import tqdm
import os
import sys
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from scipy.stats import skew, kurtosis
from scipy.signal import savgol_filter
from pathlib import Path
from collections import defaultdict

# Add project root to sys.path
PROJECT_ROOT = Path.cwd().parent.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

# Set processed data storage path (D drive for large files)
PROCESSED_DATA_ROOT = Path("D:/chanminLee/data_store/LFP_SOH_estimation")
PROCESSED_DATA_ROOT.mkdir(parents=True, exist_ok=True)

from src.data.loaders import load_hust, load_mit

## Phase 0: Data Loading

In [ ]:
DATASET_TYPE = "hust"
all_cells = load_hust(PROJECT_ROOT / "HUST_data" / "data") if DATASET_TYPE == "hust" else load_mit(PROJECT_ROOT / "MIT_data")
print(f"Loaded {len(all_cells)} cells for {DATASET_TYPE.upper()} dataset.")

In [ ]:
def save_all_cell_plots(cells_dict, output_dir):
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)
        print(f"Created directory: {output_dir}")

    print(f"Starting to save plots for {len(cells_dict)} cells...")
    
    for cell_id, cell in tqdm(cells_dict.items(), desc="Saving Plots"):
        if hasattr(cell, 'data'):
            cycle_keys = sorted(list(cell.data.keys()))
            cyc_idx = 1
            df = cell.data[cyc_idx]
            col_t = [c for c in df.columns if "Time" in c][0]
            col_v = [c for c in df.columns if "Voltage" in c][0]
            col_i = [c for c in df.columns if "Current" in c][0]
            t, v, curr = df[col_t].values, df[col_v].values, df[col_i].values
            ds_name = "HUST"
        else:
            cycle_keys = sorted(list(cell.cycles.keys()))
            cyc_idx = cycle_keys[len(cycle_keys)//2]
            cyc_data = cell.cycles[str(cyc_idx)] if str(cyc_idx) in cell.cycles else cell.cycles[cyc_idx]
            t, v, curr = cyc_data['t'], cyc_data['V'], cyc_data['I']
            ds_name = "MIT"

        row_index = np.arange(len(v))
        fig, (ax_left, ax_right) = plt.subplots(1, 2, figsize=(18, 5))

        ax_left.plot(row_index, v, color='tab:blue', label='Voltage', linewidth=1.5)
        ax_left.set_ylabel("Voltage (V)", color='tab:blue')
        ax_left.tick_params(axis='y', labelcolor='tab:blue')
        ax_left.set_xlabel("Data Row Index")
        ax_left.set_title(f"Row Index: {cell_id} (Cyc {cyc_idx})")
        ax_left.grid(True, alpha=0.2)

        ax_l_curr = ax_left.twinx()
        ax_l_curr.plot(row_index, curr, color='tab:orange', alpha=0.4, linestyle='--')
        ax_l_curr.set_ylabel("Current (A/mA)", color='tab:orange')
        ax_l_curr.tick_params(axis='y', labelcolor='tab:orange')

        ax_right.plot(t, v, color='tab:blue', linewidth=1.5)
        ax_right.set_ylabel("Voltage (V)", color='tab:blue')
        ax_right.tick_params(axis='y', labelcolor='tab:blue')
        ax_right.set_xlabel("Time (s)")
        ax_right.set_title(f"Time Scale: {cell_id} (Cyc {cyc_idx})")
        ax_right.grid(True, alpha=0.2)

        ax_r_curr = ax_right.twinx()
        ax_r_curr.plot(t, curr, color='tab:orange', alpha=0.4, linestyle='--')
        ax_r_curr.set_ylabel("Current (A/mA)", color='tab:orange')
        ax_r_curr.tick_params(axis='y', labelcolor='tab:orange')

        plt.suptitle(f"[{ds_name} Dataset] Cell {cell_id} Profiling", fontsize=14, y=1.05)
        safe_cell_id = str(cell_id).replace("/", "_")
        save_name = f"{ds_name}_{safe_cell_id}_check.png"
        plt.savefig(os.path.join(output_dir, save_name), bbox_inches='tight', dpi=150)
        plt.close(fig)

RAW_DATA_SAVE_PATH = PROJECT_ROOT / "outputs" / "figures" / "cell_checks" / DATASET_TYPE
save_all_cell_plots(all_cells, RAW_DATA_SAVE_PATH)

## Phase 1: Global Physical Cleaning & Unit Conversion

In [ ]:
CLEAN_CACHE_DIR = PROCESSED_DATA_ROOT / f"{DATASET_TYPE}_clean_chunks"
CLEAN_CACHE_DIR.mkdir(parents=True, exist_ok=True)
CLEAN_REPORT_PATH = PROJECT_ROOT / "outputs" / "cleaning_report.csv"

def clean_physical_data(df, cid, cyc, cleaning_reports):
    clean_df = df.copy()
    if "Current (mA)" in clean_df.columns:
        clean_df["Current (A)"] = clean_df["Current (mA)"] / 1000.0
    num_cols = clean_df.select_dtypes(include=[np.number]).columns
    outlier_mask = pd.Series(False, index=clean_df.index)
    detail_records = {}
    for col in num_cols:
        details = {'outlier_info': [], 'nan_info': []}
        nan_mask = clean_df[col].isna()
        if nan_mask.any():
            details['nan_info'] = clean_df.index[nan_mask].tolist()
        col_std = clean_df[col].std()
        if col_std > 1e-6:
            z_scores = np.abs((clean_df[col] - clean_df[col].mean()) / col_std)
            is_outlier = z_scores >= 7
            if is_outlier.any():
                outlier_mask |= is_outlier
                outlier_indices = clean_df.index[is_outlier].tolist()
                outlier_values = clean_df.loc[is_outlier, col].tolist()
                details['outlier_info'] = [(idx, round(val, 4)) for idx, val in zip(outlier_indices, outlier_values)]
        detail_records[col] = details
    if outlier_mask.any():
        clean_df = clean_df[~outlier_mask].copy()
    if not clean_df.empty:
        for col in num_cols:
            det = detail_records[col]
            o_count = len(det['outlier_info'])
            nan_count = len(det['nan_info'])
            if o_count > 0 or nan_count > 0:
                cleaning_reports.append({
                    "cell_id": cid, "cycle": cyc, "feature": col,
                    "outlier_removed": o_count, "interpolated_count": nan_count,
                    "outlier_details_(idx,val)": str(det['outlier_info']) if o_count > 0 else "",
                    "nan_indices": str(det['nan_info']) if nan_count > 0 else ""
                })
        clean_df[num_cols] = clean_df[num_cols].interpolate(method='linear', limit_direction='both')
        clean_df = clean_df.ffill().bfill()
        v_cols = [c for c in clean_df.columns if "Voltage" in c]
        if v_cols and len(clean_df) >= 11:
            clean_df[v_cols[0]] = savgol_filter(clean_df[v_cols[0]], window_length=11, polyorder=3)
    return clean_df

batch_groups = defaultdict(dict)
for cid, cell in all_cells.items():
    batch_id = int(str(cid).split("-")[0])
    batch_groups[batch_id][cid] = cell
all_cleaning_reports = []
for batch_id in sorted(batch_groups.keys()):
    batch_cache_path = CLEAN_CACHE_DIR / f"batch_clean_{batch_id}.pkl"
    if batch_cache_path.exists():
        report_part_path = CLEAN_CACHE_DIR / f"batch_clean_{batch_id}_report.pkl"
        if report_part_path.exists():
            with open(report_part_path, "rb") as f:
                all_cleaning_reports.extend(pickle.load(f))
        continue
    batch_cells = batch_groups[batch_id]
    batch_cleaned = {}
    batch_reports = []
    for cid, cell in batch_cells.items():
        batch_cleaned[cid] = {}
        for cyc in tqdm(list(cell.data.keys()), desc=f"  Cleaning Cell {cid}"):
            batch_cleaned[cid][cyc] = clean_physical_data(cell.data[cyc], cid, cyc, batch_reports)
    with open(batch_cache_path, "wb") as f:
        pickle.dump(batch_cleaned, f, protocol=pickle.HIGHEST_PROTOCOL)
    report_part_path = CLEAN_CACHE_DIR / f"batch_clean_{batch_id}_report.pkl"
    with open(report_part_path, "wb") as f:
        pickle.dump(batch_reports, f, protocol=pickle.HIGHEST_PROTOCOL)
    all_cleaning_reports.extend(batch_reports)
    del batch_cleaned, batch_reports
    gc.collect()
report_df = pd.DataFrame(all_cleaning_reports)
report_df.to_csv(CLEAN_REPORT_PATH, index=False)
print(f"Cleaning report saved to {CLEAN_REPORT_PATH}")

## Phase 2: Scenario-based Slicing Definition

In [ ]:
def phase2_slice_data(df, start_p=0.0, end_p=1.0, mode="C"): 
    col_i = "Current (A)" if "Current (A)" in df.columns else [c for c in df.columns if "Current" in c][0]
    mode_df = df[df[col_i] > 0.01] if mode == "C" else df[df[col_i] < -0.01]
    if mode_df.empty: return mode_df, 0, (1 if mode == "C" else 0)
    n = len(mode_df)
    s_idx, e_idx = int(n * start_p), int(n * end_p)
    sliced = mode_df.iloc[s_idx:e_idx]
    avg_pos = (start_p + end_p) / 2
    soc_label = -2 if avg_pos <= 0.3 else (-1 if avg_pos <= 0.7 else 0)
    mode_label = 1 if mode == "C" else 0
    return sliced, soc_label, mode_label

## Phase 2.1: Global Slicing Loop

In [ ]:
CLEAN_CACHE_DIR = PROCESSED_DATA_ROOT / f"{DATASET_TYPE}_clean_chunks"
CACHE_DIR = PROCESSED_DATA_ROOT / f"{DATASET_TYPE}_sliced_chunks"
CACHE_DIR.mkdir(parents=True, exist_ok=True)
scen_bounds = {"H": (0.0, 0.3), "M": (0.3, 0.7), "L": (0.7, 1.0)}
lengths = [0.1, 0.2, 0.3]
step = 0.1
batch_groups = defaultdict(dict)
for cid, cell in all_cells.items():
    batch_id = int(str(cid).split("-")[0])
    batch_groups[batch_id][cid] = cell
all_raw_segments = []
for batch_id in sorted(batch_groups.keys()):
    slice_cache_path = CACHE_DIR / f"batch_{batch_id}.pkl"
    clean_cache_path = CLEAN_CACHE_DIR / f"batch_clean_{batch_id}.pkl"
    if slice_cache_path.exists():
        with open(slice_cache_path, "rb") as f: batch_data = pickle.load(f)
        all_raw_segments.extend(batch_data['segments'])
        del batch_data
        gc.collect()
        continue
    with open(clean_cache_path, "rb") as f: batch_cleaned = pickle.load(f)
    batch_cells = batch_groups[batch_id]
    batch_segments = []
    for cid, cell in batch_cells.items():
        cleaned_cycles = batch_cleaned.get(cid, {})
        for cyc in tqdm(sorted(cleaned_cycles.keys()), desc=f"  Slicing Cell {cid}"):
            df = cleaned_cycles[cyc]
            num_cols = df.select_dtypes(include=[np.number]).columns
            df[num_cols] = df[num_cols].astype(np.float32)
            for mode in ["C", "D"]:
                for scen_name, (s_bound, e_bound) in scen_bounds.items():
                    for length in lengths:
                        max_start = e_bound - length
                        if max_start < s_bound: continue
                        for start_p in np.arange(s_bound, max_start + 0.0001, step):
                            end_p = start_p + length
                            sliced, soc_l, mode_l = phase2_slice_data(df, start_p, end_p, mode)
                            if not sliced.empty and len(sliced) >= 5:
                                batch_segments.append({
                                    "cell": cid, "cyc": cyc, "df": sliced,
                                    "soc_label": soc_l, "mode_label": mode_l,
                                    "length_p": length,
                                    "rul": cell.rul[cyc],
                                    "cycle_key": (cid, cyc)
                                })
    with open(slice_cache_path, "wb") as f: pickle.dump({'segments': batch_segments}, f, protocol=pickle.HIGHEST_PROTOCOL)
    all_raw_segments.extend(batch_segments)
    del batch_cleaned, batch_segments
    gc.collect()
print(f"Total segments: {len(all_raw_segments)}")

## Phase 2.2: Global Sliced Data Check

In [ ]:
CACHE_DIR = PROCESSED_DATA_ROOT / f"{DATASET_TYPE}_sliced_chunks"
OUTPUT_REPORT_DIR = PROJECT_ROOT / "outputs" / "reports"
OUTPUT_REPORT_DIR.mkdir(parents=True, exist_ok=True)
TXT_REPORT_PATH = OUTPUT_REPORT_DIR / f"{DATASET_TYPE}_slice_summary.txt"
PLOT_SAVE_PATH = OUTPUT_REPORT_DIR / f"{DATASET_TYPE}_slice_distributions.png"
batch_files = sorted(CACHE_DIR.glob("batch_*.pkl"))
meta_records = []
for batch_file in tqdm(batch_files, desc="Processing Batches"):
    with open(batch_file, "rb") as f: batch_data = pickle.load(f)
    for seg in batch_data['segments']:
        meta_records.append({
            "cell": seg['cell'], "cyc": seg['cyc'], "soc_label": seg['soc_label'],
            "mode_label": seg['mode_label'], "rul": seg['rul'], "seg_len": len(seg['df'])
        })
    del batch_data
    gc.collect()
meta_df = pd.DataFrame(meta_records)
meta_df['mode_str'] = meta_df['mode_label'].map({1: 'Charge', 0: 'Discharge'})
meta_df['soc_str'] = meta_df['soc_label'].astype(str)
sns.set_theme(style="whitegrid")
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
sns.countplot(data=meta_df, x='cell', hue='mode_str', ax=axes[0, 0])
sns.countplot(data=meta_df, x='soc_str', hue='mode_str', ax=axes[0, 1])
sns.histplot(data=meta_df, x='seg_len', hue='mode_str', bins=50, ax=axes[1, 0])
sns.histplot(data=meta_df, x='rul', bins=40, ax=axes[1, 1])
plt.tight_layout()
plt.savefig(PLOT_SAVE_PATH, dpi=300)
plt.show()

## Phase 3: 40D Adaptive HI Extraction Logic

In [ ]:
from scipy.stats import skew, kurtosis, pearsonr
def extract_45d_hi(df, mode="C"):
    v_col = [c for c in df.columns if "Voltage" in c][0]
    i_col = "Current (A)" if "Current (A)" in df.columns else [c for c in df.columns if "Current" in c][0]
    t_col = [c for c in df.columns if "Time" in c][0]
    temp_col = [c for c in df.columns if "Temperature" in c]
    v, i_raw, t = df[v_col].values, df[i_col].values, df[t_col].values
    i_abs = np.abs(i_raw)
    temp = df[temp_col[0]].values if temp_col else np.full_like(v, 25.0)
    dt_full = np.diff(t, prepend=t[0])
    dt_diff = np.where(np.diff(t) > 0, np.diff(t), 1e-6)
    dq_instant = (i_abs[1:] * dt_diff) / 3600.0
    dq_cum = np.cumsum(np.insert(dq_instant, 0, 0))
    his = {}
    his["c1_1_mean_v"] = np.mean(v)
    his["c1_2_var_v"] = np.var(v)
    his["c1_3_skew_v"] = skew(v) if np.std(v) > 1e-6 else 0.0
    his["c1_4_kurt_v"] = kurtosis(v) if np.std(v) > 1e-6 else 0.0
    his["c1_5_mean_t"] = np.mean(temp)
    his["c1_6_delta_t"] = np.max(temp) - np.min(temp)
    his["c1_7_dtdt"] = np.mean(np.diff(temp) / dt_diff)
    his["c1_8_mean_i"] = np.mean(i_abs)
    his["c1_9_var_i"] = np.var(i_abs)
    p_v = np.histogram(v, bins=10, density=True)[0]
    his["c1_10_ent_v"] = -np.sum(p_v * np.log(p_v + 1e-6))
    p_t = np.histogram(temp, bins=10, density=True)[0]
    his["c1_11_ent_t"] = -np.sum(p_t * np.log(p_t + 1e-6))
    his["c1_12_corr_vi"] = pearsonr(v, i_abs)[0] if np.std(v) > 1e-6 and np.std(i_abs) > 1e-6 else 0.0
    his["c1_13_corr_vt"] = pearsonr(v, temp)[0] if np.std(v) > 1e-6 and np.std(temp) > 1e-6 else 0.0
    his["c1_14_de_dq"] = np.sum(v * i_abs * dt_full) / (np.sum(i_abs * dt_full) + 1e-6)
    his["c1_15_power_var"] = np.var(v * i_raw)
    for j in range(1, 16): his[f"c2_dis_{j}"] = his[f"c3_cha_{j}"] = 0.0
    dvdt = np.diff(v) / dt_diff
    d2vdt2 = np.diff(dvdt) / np.where(dt_diff[1:] > 0, dt_diff[1:], 1e-6) if len(dvdt) > 1 else np.array([0.0])
    if mode == "D":
        his["c2_dis_1"] = np.mean(dvdt)
        his["c2_dis_2"] = np.var(dvdt)
        his["c2_dis_3"] = np.mean(d2vdt2)
        his["c2_dis_4"] = np.sum((np.max(v) - v) * dt_full)
        di = np.diff(i_raw)
        valid_di = np.abs(di) > 0.005
        his["c2_dis_5"] = np.mean(np.abs(np.diff(v)[valid_di] / di[valid_di])) if np.any(valid_di) else 0.0
        his["c2_dis_6"] = np.sum(v * i_abs * dt_full) / 3600.0
        his["c2_dis_7"] = np.sum(i_abs * dt_full) / 3600.0
        dvdq = np.diff(v) / np.where(dq_instant > 0, dq_instant, 1e-6)
        his["c2_dis_8"] = np.mean(dvdq)
        his["c2_dis_9"] = np.var(dvdq)
        his["c2_dis_10"] = np.min(v) / (np.max(v) + 1e-6)
        his["c2_dis_11"] = (temp[-1] - temp[0]) / (his["c2_dis_7"] + 1e-6)
        his["c2_dis_12"] = np.sum(np.abs(np.diff(dvdt)))
        his["c2_dis_13"] = np.sum(v * dt_full) / (np.mean(i_abs) + 1e-6)
        if len(dq_cum) > 1: his["c2_dis_14"] = np.polyfit(dq_cum, v, 1)[0]
        his["c2_dis_15"] = np.sum(v * dq_cum) / (np.sum(dq_cum) + 1e-6)
    elif mode == "C":
        his["c3_cha_1"] = np.mean(dvdt)
        his["c3_cha_2"] = np.var(dvdt)
        his["c3_cha_3"] = np.mean(d2vdt2)
        his["c3_cha_4"] = np.sum((v - np.min(v)) * dt_full)
        dv_pos = np.where(np.diff(v) > 0.0005, np.diff(v), 1e-6)
        dqdv = dq_instant / dv_pos
        his["c3_cha_5"] = np.mean(dqdv)
        his["c3_cha_6"] = np.var(dqdv)
        his["c3_cha_7"] = np.sum(v * i_abs * dt_full) / 3600.0
        his["c3_cha_8"] = np.sum(i_abs * dt_full) / 3600.0
        his["c3_cha_9"] = np.max(v) / (np.min(v) + 1e-6)
        his["c3_cha_10"] = (temp[-1] - temp[0]) / (his["c3_cha_8"] + 1e-6)
        his["c3_cha_11"] = np.sum(np.abs(np.diff(dvdt)))
        his["c3_cha_12"] = np.mean(i_abs) / (np.abs(np.mean(dvdt)) + 1e-6)
        p_dvdt = np.histogram(dvdt, bins=10, density=True)[0]
        his["c3_cha_13"] = -np.sum(p_dvdt * np.log(p_dvdt + 1e-6))
        his["c3_cha_14"] = np.max(dqdv) if len(dqdv) > 0 else 0.0
        his["c3_cha_15"] = np.sum((v[1:] - v[0]) * dq_instant)
    return pd.Series(his)

## Phase 4: Feature Extraction & Scaling

In [ ]:
FEATURE_BATCH_DIR = PROCESSED_DATA_ROOT / f"{DATASET_TYPE}_feature_batches"
FEATURE_BATCH_DIR.mkdir(parents=True, exist_ok=True)
SCALER_CACHE_PATH = PROCESSED_DATA_ROOT / f"{DATASET_TYPE}_scaler.pkl"
batch_files = sorted(CACHE_DIR.glob("batch_*.pkl"))
scaler = StandardScaler()
total_items_extracted = 0
for batch_file in batch_files:
    batch_id = batch_file.stem
    with open(batch_file, "rb") as f: batch_data = pickle.load(f)
    batch_feature_pool = []
    for item in tqdm(batch_data['segments'], desc=f"  Extracting [{batch_id}]"):
        hi = extract_45d_hi(item['df'], "C" if item['mode_label'] == 1 else "D")
        batch_feature_pool.append({
            "cell": item['cell'], "cyc": item['cyc'], "raw_x": hi,
            "soc_label": item['soc_label'], "mode_label": item['mode_label'],
            "length_p": item['length_p'], "y": item['rul']
        })
    raw_x_batch = [x['raw_x'] for x in batch_feature_pool]
    scaler.partial_fit(raw_x_batch)
    with open(FEATURE_BATCH_DIR / f"raw_{batch_id}.pkl", "wb") as f: pickle.dump(batch_feature_pool, f)
    total_items_extracted += len(batch_feature_pool)
    del batch_data, batch_feature_pool; gc.collect()
for raw_file in sorted(FEATURE_BATCH_DIR.glob("raw_batch_*.pkl")):
    batch_id = raw_file.stem.replace("raw_", "")
    with open(raw_file, "rb") as f: batch_pool = pickle.load(f)
    raw_x_batch = [x['raw_x'] for x in batch_pool]
    scaled_x_batch = scaler.transform(raw_x_batch)
    for i, item in enumerate(batch_pool): item['scaled_x'] = scaled_x_batch[i]
    with open(FEATURE_BATCH_DIR / f"final_{batch_id}.pkl", "wb") as f: pickle.dump(batch_pool, f)
    raw_file.unlink()
with open(SCALER_CACHE_PATH, "wb") as f: pickle.dump(scaler, f)
print("Extraction and Batch Scaling complete.")

## Phase 5: Final Saving

In [ ]:
if 'feature_pool' not in locals() and 'feature_pool' not in globals():
    FEATURE_BATCH_DIR = PROCESSED_DATA_ROOT / f"{DATASET_TYPE}_feature_batches"
    batch_files = sorted(FEATURE_BATCH_DIR.glob("final_*.pkl"))
    feature_pool = []
    for bf in tqdm(batch_files, desc="Loading Final Batches"):
        with open(bf, "rb") as f: feature_pool.extend(pickle.load(f))
def save_processed_dataset(pool, dataset_type):
    out_path = PROCESSED_DATA_ROOT / f"{dataset_type}_optimized_tensors.pkl"
    final_data = []
    for item in pool:
        final_data.append({
            "cell": item['cell'], "cyc": item['cyc'], "x": item['scaled_x'],
            "soc_label": item['soc_label'], "mode_label": item['mode_label'],
            "length_p": item['length_p'], "y": item.get('y', item.get('rul'))
        })
    with open(out_path, "wb") as f: pickle.dump(final_data, f)
    print(f"Saved to {out_path}")
save_processed_dataset(feature_pool, DATASET_TYPE)

## Phase 6: Visualization

In [ ]:
from sklearn.decomposition import PCA
def analyze_final_hi_dataset(dataset_type):
    data_path = PROCESSED_DATA_ROOT / f"{dataset_type}_optimized_tensors.pkl"
    with open(data_path, "rb") as f: data = pickle.load(f)
    df = pd.DataFrame(data)
    x_expanded = pd.DataFrame(df['x'].tolist())
    x_expanded.columns = [f"HI_{i}" for i in range(x_expanded.shape[1])]
    df = pd.concat([df.drop('x', axis=1), x_expanded], axis=1)
    df['batch_id'] = df['cell'].apply(lambda x: str(x).split('-')[0])
    df['scen_str'] = df['soc_label'].map({-2: 'High SOC', -1: 'Mid SOC', 0: 'Low SOC'})
    df['mode_str'] = df['mode_label'].map({1: 'Charge', 0: 'Discharge'})
    fig = plt.figure(figsize=(22, 18))
    ax1 = plt.subplot(2, 2, 1)
    df_melt = df.melt(id_vars=['scen_str'], value_vars=[f"HI_{i}" for i in range(5)])
    sns.boxplot(data=df_melt, x='variable', y='value', hue='scen_str', ax=ax1)
    ax2 = plt.subplot(2, 2, 2)
    sns.violinplot(data=df, x='batch_id', y='HI_0', hue='mode_str', split=True, ax=ax2)
    ax3 = plt.subplot(2, 2, 3)
    pca = PCA(n_components=2)
    hi_cols = [f"HI_{i}" for i in range(x_expanded.shape[1])]
    if df[hi_cols].isnull().any().any():
        nan_count = df[hi_cols].isnull().any(axis=1).sum()
        print(f"Warning: Found {nan_count} rows with NaN values. Dropping them for PCA.")
        pca_df = df.dropna(subset=hi_cols)
    else:
        pca_df = df
    pca_res = pca.fit_transform(pca_df[hi_cols])
    plt.scatter(pca_res[:, 0], pca_res[:, 1], c=pca_df['y'], cmap='viridis_r', s=10)
    plt.colorbar(label='RUL')
    ax4 = plt.subplot(2, 2, 4)
    df[[f"HI_{i}" for i in range(x_expanded.shape[1])] + ['y']].corr()['y'].abs().sort_values(ascending=False).iloc[1:21].plot(kind='bar', ax=ax4)
    plt.tight_layout()
    plt.show()

    # --- Additional Discharge Analysis Plots ---
    dis_df = df[df['mode_label'] == 0].copy()
    if not dis_df.empty:
        feat_cols = [f"HI_{i}" for i in range(30)]
        melted = dis_df.melt(id_vars=['length_p', 'HI_7'], value_vars=feat_cols)
        melted['length_pct'] = (melted['length_p'] * 100).astype(int)
        
        fig, (ax_ext1, ax_ext2) = plt.subplots(2, 1, figsize=(20, 24))
        
        # 1. Distribution by Length (10, 20, 30)
        sns.boxplot(data=melted, x='value', y='variable', hue='length_pct', ax=ax_ext1)
        ax_ext1.set_title("Feature Distributions by Slice Length (10%, 20%, 30%) - Discharge")
        ax_ext1.grid(True, alpha=0.3)
        
        # 2. vs Discharge C-rate Mean (HI_7)
        sns.scatterplot(data=melted, x='HI_7', y='variable', hue='value', palette='viridis', alpha=0.5, ax=ax_ext2)
        ax_ext2.set_title("Features vs Mean Discharge Current (C-rate proxy) - Discharge")
        ax_ext2.grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.show()
analyze_final_hi_dataset(DATASET_TYPE)